<div class="blog-language-switch" role="group" aria-label="Article language">
<span aria-current="page">English</span>
<a href="/ipynb/zh-CN/Computer-Science/Data-Structures-and-Algorithms/06-searching-sorting-divide-conquer.html" lang="zh-CN" hreflang="zh-CN">中文</a>
</div>

[Back to Data Structures and Algorithms guideline](Data-Structure&Algorithm.html)


## **Searching, Sorting, and Divide-and-Conquer** {#searching-sorting-and-divide-and-conquer}

Searching and sorting are not only library operations. They expose several ideas used throughout algorithm design: maintaining a shrinking candidate set, preserving loop invariants, rearranging data under an explicit contract, decomposing a problem into smaller instances, and solving the resulting recurrence.

This chapter connects those ideas in one progression:

- binary search turns sorted order or a monotone predicate into logarithmic candidate elimination;
- sorting algorithms reveal trade-offs among comparisons, writes, stability, auxiliary memory, and worst-case guarantees;
- merge sort, quicksort, and heapsort obtain $O(n\log n)$ behavior through different structural invariants;
- divide-and-conquer separates recursive subproblem work from division and combination work;
- recurrence expansion and the Master Theorem explain when recursion is efficient;
- quickselect shows that finding one rank does not require fully sorting both sides.

Python's built-in <code>sorted</code> and <code>list.sort</code> should remain the practical default for ordinary application code. The implementations here make the hidden invariants and costs visible so that the same reasoning can be transferred to less familiar problems.


### **Binary Search** {#binary-search}

**Binary search** locates a target in a sorted sequence by comparing the target with the middle value and discarding the half that cannot contain it. It resembles looking up a word in a dictionary: the current page tells whether every earlier or every later page can be ignored.

The sorted-order precondition is what makes elimination safe. On an unsorted array, observing that the middle value is smaller than the target says nothing about values to its left. Binary search is better than linear search only when order is already available, can be built once and reused, or comes naturally from the problem.

Using a closed candidate interval <code>[low, high]</code>, the loop invariant is:

> If the target occurs in the sequence, at least one occurrence remains between <code>low</code> and <code>high</code>, inclusive.

~~~text
BINARY-SEARCH(values, target)
    low <- 0
    high <- length(values) - 1

    while low <= high
        mid <- low + floor((high - low) / 2)
        if values[mid] equals target
            return mid
        if values[mid] < target
            low <- mid + 1
        else
            high <- mid - 1
    return not-found
~~~

![Binary search compares the middle value, discards an impossible half, and preserves the target candidate interval.](assets/binary-search-steps.svg){fig-align="center" width="96%"}

The midpoint expression

$$
\operatorname{mid}=\operatorname{low}+\left\lfloor\frac{\operatorname{high}-\operatorname{low}}{2}\right\rfloor
$$

chooses the lower middle index and avoids the integer overflow that <code>(low + high) // 2</code> can cause in fixed-width languages. Python integers do not overflow, but the safer form communicates the invariant correctly across languages.

After <code>k</code> unsuccessful comparisons, at most $n/2^k$ candidates remain. The loop ends when this quantity falls below 1, so $k$ is $O(\log_2 n)$. Time is therefore $O(\log n)$ and iterative auxiliary space is $O(1)$. A recursive implementation has the same comparisons but uses $O(\log n)$ call-stack space.

Binary search is useful for exact lookup, dictionary indexes, version boundaries, threshold decisions, and answer-space optimization. Random access matters: applying index-based binary search to a linked list loses the $O(1)$ middle access that makes the array version efficient.

<details>
<summary>Python implementation: exact binary search with invariant checks</summary>

~~~python
from collections.abc import Sequence


def binary_search(values: Sequence[int], target: int) -> int | None:
    """Return an index containing target, or None if it is absent."""
    low = 0
    high = len(values) - 1

    while low <= high:
        mid = low + (high - low) // 2

        if values[mid] == target:
            return mid
        if values[mid] < target:
            # Sorted order proves every index through mid is too small.
            low = mid + 1
        else:
            # Sorted order proves every index from mid onward is too large.
            high = mid - 1

    return None


numbers = [2, 5, 8, 12, 16, 23, 38]
assert binary_search(numbers, 23) == 5
assert binary_search(numbers, 2) == 0
assert binary_search(numbers, 40) is None
assert binary_search([], 7) is None
~~~

</details>

**Practice.** [LeetCode 704 - Binary Search](https://leetcode.com/problems/binary-search/) is the direct closed-interval implementation exercise and is useful for checking off-by-one behavior at both ends.


### **Boundary Search Templates** {#boundary-search-templates}

Many problems do not ask whether one exact value exists. They ask for the first position meeting a condition, the first occurrence of a duplicate, the insertion point of a value, or the smallest feasible answer. These are **boundary searches** over a monotone predicate.

Suppose <code>predicate(i)</code> is false for an initial region and true from one boundary onward:

$$
F,F,\ldots,F,T,T,\ldots,T.
$$

The goal is to find the first true index. A half-open interval <code>[low, high)</code> is convenient because its size is exactly <code>high - low</code> and an empty interval has <code>low == high</code>. The invariant is:

- every index below <code>low</code> is known false;
- every index at or above <code>high</code> is known true;
- the first true position remains inside <code>[low, high]</code>.

~~~text
FIRST-TRUE(low, high, predicate)       // search [low, high)
    while low < high
        mid <- low + floor((high - low) / 2)
        if predicate(mid) is true
            high <- mid                // mid may be the first true index
        else
            low <- mid + 1             // mid is definitely too early
    return low
~~~

![A half-open boundary search shrinks a false-then-true predicate until low equals high at the first true index.](assets/boundary-search-invariant.svg){fig-align="center" width="96%"}

The asymmetric updates are essential. When the predicate is true, <code>mid</code> remains a candidate, so assign <code>high = mid</code>. When false, <code>mid</code> cannot be the answer, so assign <code>low = mid + 1</code>. Both updates strictly reduce <code>high - low</code>, proving termination.

For a sorted array, <code>lower_bound(target)</code> is the first index whose value is at least the target. <code>upper_bound(target)</code> is the first index whose value is greater than the target. Their difference counts duplicate occurrences. The same template can search an answer domain, such as the smallest speed, capacity, or deadline for which a feasibility test becomes true.

The search makes $O(\log R)$ predicate calls for a domain containing <code>R</code> candidate positions. If one predicate evaluation costs $P$, total time is $O(P\log R)$, not merely $O(\log R)$. Auxiliary space remains $O(1)$ for an iterative implementation.

<details>
<summary>Python implementation: first true, lower bound, and upper bound</summary>

~~~python
from collections.abc import Callable, Sequence


def first_true(low: int, high: int, predicate: Callable[[int], bool]) -> int:
    """Return the first true index in the half-open domain [low, high)."""
    while low < high:
        mid = low + (high - low) // 2
        if predicate(mid):
            high = mid                 # Preserve mid as a possible boundary.
        else:
            low = mid + 1
    return low


def lower_bound(values: Sequence[int], target: int) -> int:
    return first_true(0, len(values), lambda i: values[i] >= target)


def upper_bound(values: Sequence[int], target: int) -> int:
    return first_true(0, len(values), lambda i: values[i] > target)


values = [1, 2, 2, 2, 5, 8]
assert lower_bound(values, 2) == 1
assert upper_bound(values, 2) == 4
assert upper_bound(values, 2) - lower_bound(values, 2) == 3
assert lower_bound(values, 7) == 5
assert lower_bound(values, 10) == len(values)
~~~

</details>

**Practice.** [LeetCode 34 - Find First and Last Position of Element in Sorted Array](https://leetcode.com/problems/find-first-and-last-position-of-element-in-sorted-array/) requires two boundary searches rather than stopping at an arbitrary matching duplicate.


### **Sorting Properties and Lower Bounds** {#sorting-properties-and-lower-bounds}

A sorting algorithm must satisfy two correctness conditions. Its output keys are in nondecreasing order, and its output is a **permutation** of the input: no element is lost, duplicated, or invented. Real applications often require additional properties that are invisible when every value is distinct.

- A **stable** sort preserves the original relative order of records with equal keys. Stability enables multi-stage sorting: sorting by a secondary key and then stably by a primary key keeps secondary order inside equal-primary groups.
- An **in-place** sort uses only constant or logarithmic auxiliary storage beyond the input array. The precise definition should state whether the recursion stack counts.
- An **adaptive** sort becomes faster when the input already contains useful order. Insertion sort and Timsort exploit existing runs; selection sort performs essentially the same comparisons regardless of order.
- An **online** sort can incorporate items as they arrive, while an offline algorithm expects the complete collection.
- Worst-case, average-case, and expected complexity are different promises. Expected randomized quicksort is not the same guarantee as worst-case heapsort.

![Sorting has observable stability and memory properties, while a comparison decision tree imposes an Omega(n log n) worst-case lower bound.](assets/sorting-contract-lower-bound.svg){fig-align="center" width="96%"}

Why can no comparison sort guarantee $o(n\log n)$ comparisons for arbitrary distinct keys? There are $n!$ possible input permutations. A deterministic comparison algorithm is a binary decision tree: each comparison has at most two outcomes, and each permutation must reach a distinguishable leaf. A tree of height <code>h</code> has at most $2^h$ leaves, so

$$
2^h \ge n!,\qquad h\ge \log_2(n!).
$$

Using Stirling's approximation, $\log_2(n!)=\Theta(n\log n)$. Therefore every comparison-based sort requires $\Omega(n\log n)$ comparisons in the worst case. Merge sort and heapsort match this asymptotic bound.

The restriction “comparison-based” matters. Counting sort and radix sort use key structure rather than only pairwise comparisons and can run in $O(n+k)$ or related bounds, where <code>k</code> describes the key range or number of digits. They do not contradict the decision-tree lower bound because one operation can reveal more than a binary comparison outcome.

<details>
<summary>Python implementation: verify order, permutation, and stability</summary>

~~~python
from collections import Counter, defaultdict
from dataclasses import dataclass


@dataclass(frozen=True)
class Record:
    key: int
    label: str                    # Distinguishes equal-key records.


def verify_stable_sort(original: list[Record], result: list[Record]) -> bool:
    """Check the three observable parts of a stable sorting contract."""
    # 1. The result must contain exactly the same records.
    if Counter(original) != Counter(result):
        return False

    # 2. Keys must be nondecreasing.
    if any(result[i - 1].key > result[i].key for i in range(1, len(result))):
        return False

    # 3. Labels inside each equal-key group must preserve input order.
    original_groups: dict[int, list[str]] = defaultdict(list)
    result_groups: dict[int, list[str]] = defaultdict(list)
    for record in original:
        original_groups[record.key].append(record.label)
    for record in result:
        result_groups[record.key].append(record.label)
    return original_groups == result_groups


records = [Record(2, "A"), Record(1, "X"),
           Record(2, "B"), Record(1, "Y")]
stable_result = [Record(1, "X"), Record(1, "Y"),
                 Record(2, "A"), Record(2, "B")]
unstable_result = [Record(1, "Y"), Record(1, "X"),
                   Record(2, "A"), Record(2, "B")]

assert verify_stable_sort(records, stable_result)
assert not verify_stable_sort(records, unstable_result)
~~~

</details>

**Practice.** [LeetCode 1122 - Relative Sort Array](https://leetcode.com/problems/relative-sort-array/) encourages thinking beyond “ascending order” by defining a custom ordering contract and allowing counting-based alternatives.


### **Elementary Sorting Algorithms** {#elementary-sorting-algorithms}

Selection sort, insertion sort, and bubble sort all have $O(n^2)$ worst-case time, but they perform different work and are useful under different constraints.

**Selection sort** repeatedly finds the minimum of the unsorted suffix and swaps it into the next output position. After iteration <code>i</code>, the prefix through <code>i</code> contains the globally smallest <code>i + 1</code> values in final positions. It always performs $\Theta(n^2)$ comparisons but only $O(n)$ swaps, making it relevant when writes are unusually expensive.

**Insertion sort** maintains a sorted prefix and inserts the next value into its correct position by shifting larger values right. It is stable when equal values are not shifted past each other. Its running time is $\Theta(n+I)$, where <code>I</code> is the number of inversions, so nearly sorted input can approach linear time.

**Bubble sort** repeatedly swaps adjacent inversions. After one complete left-to-right pass, the largest remaining value reaches the end. An early-stop flag gives $O(n)$ best-case time on already sorted input, but insertion sort is generally the more useful adaptive elementary algorithm.

~~~text
SELECTION-SORT(A)
    for boundary from 0 to n - 2
        minimum <- index of smallest item in A[boundary:n]
        swap A[boundary] and A[minimum]

INSERTION-SORT(A)
    for i from 1 to n - 1
        value <- A[i]
        shift larger prefix values one position right
        insert value into the created gap

BUBBLE-SORT(A)
    repeat passes over adjacent pairs
        swap every inverted pair
        stop if a pass made no swap
~~~

::: {layout-ncol=3}
![Selection sort repeatedly selects the next minimum.](assets/selection-sort.gif){width="92%" fig-align="center"}

![Insertion sort grows a sorted prefix by shifting values.](assets/insertion-sort.gif){width="100%" fig-align="center"}

![Bubble sort moves large values right through adjacent swaps.](assets/bubble-sort.gif){width="92%" fig-align="center"}
:::

*Open visual sources: [Selection sort animation](https://commons.wikimedia.org/wiki/File:Selection_sort_animation.gif), [AnimazioneInsertionSort.gif](https://commons.wikimedia.org/wiki/File:AnimazioneInsertionSort.gif), and [Sorting bubblesort anim.gif](https://commons.wikimedia.org/wiki/File:Sorting_bubblesort_anim.gif) (public-domain / CC BY-SA).*

| Algorithm | Best time | Average / worst | Stable | In-place | Characteristic cost |
|---|---:|---:|---|---|---|
| selection sort | $\Theta(n^2)$ | $\Theta(n^2)$ | usually no | yes | few swaps, many comparisons |
| insertion sort | $\Theta(n)$ | $\Theta(n^2)$ | yes | yes | adaptive to few inversions |
| bubble sort with early stop | $\Theta(n)$ | $\Theta(n^2)$ | yes | yes | many adjacent swaps |

<details>
<summary>Python implementation: selection, insertion, and bubble sort</summary>

~~~python
def selection_sort(values: list[int]) -> None:
    for boundary in range(len(values) - 1):
        minimum = boundary
        for index in range(boundary + 1, len(values)):
            if values[index] < values[minimum]:
                minimum = index
        values[boundary], values[minimum] = values[minimum], values[boundary]


def insertion_sort(values: list[int]) -> None:
    for index in range(1, len(values)):
        value = values[index]
        position = index

        # Shift only strictly larger values, preserving stable equal order.
        while position > 0 and values[position - 1] > value:
            values[position] = values[position - 1]
            position -= 1
        values[position] = value


def bubble_sort(values: list[int]) -> None:
    for end in range(len(values) - 1, 0, -1):
        swapped = False
        for index in range(end):
            if values[index] > values[index + 1]:
                values[index], values[index + 1] = (
                    values[index + 1], values[index]
                )
                swapped = True
        if not swapped:                    # Remaining prefix is already sorted.
            return


for algorithm in (selection_sort, insertion_sort, bubble_sort):
    sample = [5, 2, 4, 6, 1, 3, 2]
    algorithm(sample)
    assert sample == [1, 2, 2, 3, 4, 5, 6]
~~~

</details>

**Practice.** [LeetCode 147 - Insertion Sort List](https://leetcode.com/problems/insertion-sort-list/) transfers the sorted-prefix invariant from an array to linked-list pointer operations.


### **Merge Sort** {#merge-sort}

**Merge sort** divides an array by position into two halves, recursively sorts both halves, and merges the two sorted results. Its key advantage is predictability: every input arrangement receives the same $\Theta(n\log n)$ asymptotic treatment.

The merge step maintains two indices, one for each sorted run. The smaller front value is the smallest value not yet emitted, so it can be copied safely to the output. When values compare equal, taking the left value first preserves original cross-half order and makes the algorithm stable.

~~~text
MERGE-SORT(values)
    if length(values) <= 1
        return values
    split values into left and right halves
    left <- MERGE-SORT(left)
    right <- MERGE-SORT(right)
    return MERGE(left, right)

MERGE(left, right)
    repeatedly copy the smaller current value
    copy any remaining suffix after one side is exhausted
~~~

![Merge sort splits by position until singleton base cases, then merges adjacent sorted runs.](assets/merge-sort-recursion.svg){fig-align="center" width="96%"}

![Animated merge sort progressively combines values into sorted runs.](assets/merge-sort.gif){fig-align="center" width="44%"}

*Open animation source: [Wikimedia Commons - Merge sort animation2.gif](https://commons.wikimedia.org/wiki/File:Merge_sort_animation2.gif) (CC BY-SA 2.5).*

For an input of size <code>n</code>, merge sort creates two subproblems of size approximately <code>n/2</code> and performs $\Theta(n)$ merge work:

$$
T(n)=2T(n/2)+\Theta(n).
$$

There are $\Theta(\log n)$ split levels, and each merge level processes all <code>n</code> values once, giving $\Theta(n\log n)$ time in the best, average, and worst cases. Standard array merge sort uses $\Theta(n)$ auxiliary array storage plus $O(\log n)$ recursion stack. It is not in-place under the usual practical definition.

Merge sort is especially effective for linked lists, where merging changes links without an auxiliary value array, and for external sorting, where sequentially merging large disk runs is more efficient than random access. It also supports algorithmic augmentation: inversion counting and range statistics can be accumulated during merge.

<details>
<summary>Python implementation: stable top-down merge sort</summary>

~~~python
from collections.abc import Sequence


def merge_sort(values: Sequence[int]) -> list[int]:
    """Return a new stably sorted list."""
    if len(values) <= 1:
        return list(values)

    middle = len(values) // 2
    left = merge_sort(values[:middle])
    right = merge_sort(values[middle:])

    merged: list[int] = []
    left_index = 0
    right_index = 0

    while left_index < len(left) and right_index < len(right):
        if left[left_index] <= right[right_index]:
            # Taking the left item on equality preserves stability.
            merged.append(left[left_index])
            left_index += 1
        else:
            merged.append(right[right_index])
            right_index += 1

    # Exactly one of these suffixes can be nonempty.
    merged.extend(left[left_index:])
    merged.extend(right[right_index:])
    return merged


assert merge_sort([8, 3, 2, 9, 7, 1, 5, 4]) == [1, 2, 3, 4, 5, 7, 8, 9]
assert merge_sort([]) == []
~~~

</details>

**Practice.** [LeetCode 148 - Sort List](https://leetcode.com/problems/sort-list/) is a natural merge-sort exercise because linked-list halves can be split with slow/fast pointers and merged by relinking nodes.


### **Quick Sort** {#quick-sort}

**Quicksort** chooses a pivot, partitions values so smaller keys lie on one side and larger keys on the other, then recursively sorts those outer regions. Unlike merge sort, it performs most structural work before recursion and usually rearranges the input array in place.

The pivot does not need to be the median for correctness. Pivot quality determines recursion balance and therefore performance. A pivot near the median creates two subproblems of roughly half size; repeatedly choosing an extreme value creates one subproblem of size <code>n - 1</code> and leads to quadratic time.

A three-way partition is robust when many values equal the pivot. During scanning it maintains:

- <code>[low, lt)</code>: values smaller than the pivot;
- <code>[lt, scan)</code>: values equal to the pivot;
- <code>[scan, gt]</code>: not yet classified;
- <code>(gt, high]</code>: values greater than the pivot.

~~~text
THREE-WAY-PARTITION(A, low, high, pivot)
    lt <- low; scan <- low; gt <- high
    while scan <= gt
        if A[scan] < pivot
            swap A[lt] and A[scan]; increment lt and scan
        else if A[scan] > pivot
            swap A[scan] and A[gt]; decrement gt
        else
            increment scan
    return lt, gt

QUICKSORT(A, low, high)
    choose a pivot
    lt, gt <- THREE-WAY-PARTITION(A, low, high, pivot)
    QUICKSORT(A, low, lt - 1)
    QUICKSORT(A, gt + 1, high)
~~~

![Three-way partition fixes the equal-pivot region and recurses only on smaller and larger values.](assets/quicksort-partition.svg){fig-align="center" width="96%"}

![Quicksort animation: the highlighted pivot separates values before recursive sorting continues.](assets/quicksort.gif){fig-align="center" width="48%"}

*Open animation source: [Wikimedia Commons - Sorting quicksort anim.gif](https://commons.wikimedia.org/wiki/File:Sorting_quicksort_anim.gif) (CC BY-SA / GFDL).*

Balanced partitions satisfy $T(n)=2T(n/2)+\Theta(n)=\Theta(n\log n)$. Extreme partitions satisfy $T(n)=T(n-1)+\Theta(n)=\Theta(n^2)$. Choosing pivots uniformly at random gives expected $\Theta(n\log n)$ time against input order, while the theoretical worst case remains $\Theta(n^2)$.

In-place partitioning uses $O(1)$ local storage. The recursion stack is expected $O(\log n)$ under balanced randomized behavior but can reach $O(n)$ in the worst case. Quicksort is normally unstable because swaps can reverse equal-key records. Its strong cache locality and small constants often make it fast for arrays despite the weaker worst-case guarantee.

<details>
<summary>Python implementation: randomized in-place three-way quicksort</summary>

~~~python
from random import Random


def quicksort(values: list[int], seed: int = 0) -> None:
    """Sort in place with randomized pivots and three-way partitioning."""
    random = Random(seed)

    def sort(low: int, high: int) -> None:
        if low >= high:
            return

        pivot = values[random.randrange(low, high + 1)]
        less = low
        scan = low
        greater = high

        while scan <= greater:
            if values[scan] < pivot:
                values[less], values[scan] = values[scan], values[less]
                less += 1
                scan += 1
            elif values[scan] > pivot:
                values[scan], values[greater] = values[greater], values[scan]
                greater -= 1
                # The new value at scan has not been classified yet.
            else:
                scan += 1

        sort(low, less - 1)
        sort(greater + 1, high)

    sort(0, len(values) - 1)


sample = [7, 2, 5, 3, 9, 5, 1, 5]
quicksort(sample, seed=42)
assert sample == [1, 2, 3, 5, 5, 5, 7, 9]
~~~

</details>

**Practice.** [LeetCode 912 - Sort an Array](https://leetcode.com/problems/sort-an-array/) is a suitable environment for testing pivot choice, duplicate handling, and recursion depth under adversarial cases.


### **Heap Sort** {#heap-sort}

**Heapsort** treats the input array as a max-heap. The largest value occupies the root at index 0. Swapping that root with the final element places one value into its permanent sorted position; reducing the logical heap size excludes the sorted suffix, and sift-down repairs the remaining heap prefix.

This section focuses on heapsort's sorting contract. The complete-tree representation, index formulas, sift operations, and bottom-up $O(n)$ heap construction are developed fully in [Binary Heaps and Heapify](04-trees-heaps-priority-queues.html#binary-heaps-and-heapify).

~~~text
HEAPSORT(A)
    build a max-heap across the full array
    for end from length(A) - 1 down to 1
        swap A[0] and A[end]
        reduce logical heap size to end
        sift the new root downward inside the heap prefix
~~~

![Heapsort stores a shrinking max-heap and a growing sorted suffix inside the same array.](assets/heapsort-inplace-phases.svg){fig-align="center" width="96%"}

Bottom-up heap construction costs $O(n)$. The algorithm then performs <code>n - 1</code> root extractions, each requiring at most $O(\log n)$ sift work, so best, average, and worst time are all $O(n\log n)$. Iterative heapsort uses $O(1)$ auxiliary storage and gives a stronger worst-case memory and time guarantee than ordinary quicksort.

The trade-off is practical locality and stability. Sift-down jumps between parent and child indices rather than scanning contiguous runs, so it is often slower than well-engineered quicksort on in-memory arrays. Long-distance swaps make the algorithm unstable. Heapsort is appropriate when worst-case $O(n\log n)$ and in-place storage matter more than stability and typical constant factors.

<details>
<summary>Python implementation: in-place max-heap sort</summary>

~~~python
def heapsort(values: list[int]) -> None:
    def sift_down(root: int, heap_size: int) -> None:
        while True:
            left = 2 * root + 1
            right = left + 1
            largest = root

            if left < heap_size and values[left] > values[largest]:
                largest = left
            if right < heap_size and values[right] > values[largest]:
                largest = right
            if largest == root:
                return

            values[root], values[largest] = values[largest], values[root]
            root = largest

    # Bottom-up heapify: leaves already satisfy the heap property.
    for root in range(len(values) // 2 - 1, -1, -1):
        sift_down(root, len(values))

    # Move each maximum to the excluded sorted suffix.
    for end in range(len(values) - 1, 0, -1):
        values[0], values[end] = values[end], values[0]
        sift_down(0, end)


sample = [12, 11, 13, 5, 6, 7, 6]
heapsort(sample)
assert sample == [5, 6, 6, 7, 11, 12, 13]
~~~

</details>

**Practice.** [LeetCode 912 - Sort an Array](https://leetcode.com/problems/sort-an-array/) can also be used to compare heapsort's guaranteed recursion-free behavior with randomized quicksort.


### **Divide-and-Conquer Design** {#divide-and-conquer-design}

**Divide-and-conquer** solves a problem by dividing it into smaller instances of the same problem, solving those instances recursively, and combining their answers. The pattern is valuable when subproblems are substantially smaller and their results summarize enough information for an efficient combination step.

~~~text
SOLVE(problem of size n)
    if n is small enough
        return direct base-case answer
    subproblems <- DIVIDE(problem)
    partial_answers <- recursively SOLVE each required subproblem
    return COMBINE(partial_answers)
~~~

![The divide-and-conquer template separates recursive subproblem count and size from nonrecursive combine work.](assets/divide-conquer-template.svg){fig-align="center" width="96%"}

If a problem creates <code>a</code> subproblems, each of size <code>n/b</code>, and division plus combination costs $f(n)$, its recurrence often has the form

$$
T(n)=aT(n/b)+f(n).
$$

Here <code>a</code> is the number of recursive branches, <code>b</code> is the shrink factor, and $f(n)$ includes work outside recursive calls. Binary search has <code>a = 1</code>, <code>b = 2</code>, and $f(n)=\Theta(1)$. Merge sort has <code>a = 2</code>, <code>b = 2</code>, and $f(n)=\Theta(n)$.

Divide-and-conquer is not automatically efficient. Creating overlapping subproblems may repeat enormous work and instead call for memoization or dynamic programming. Highly unbalanced subproblems can create deep recursion. A costly combine step may dominate all recursive savings. The design should therefore state base cases, prove progress toward them, identify whether subproblems overlap, and analyze combination cost.

As an example, the maximum-subarray problem can split at the midpoint. The best subarray lies entirely in the left half, entirely in the right half, or crosses the midpoint. The crossing answer is built from the best suffix of the left half and best prefix of the right half.

<details>
<summary>Python implementation: divide-and-conquer maximum subarray</summary>

~~~python
def maximum_subarray_divide_conquer(values: list[int]) -> int:
    """Return the maximum sum of a nonempty contiguous subarray."""
    if not values:
        raise ValueError("values must be nonempty")

    def solve(low: int, high: int) -> int:
        if high - low == 1:
            return values[low]

        middle = low + (high - low) // 2
        best_left = solve(low, middle)
        best_right = solve(middle, high)

        # Best suffix ending immediately before the midpoint.
        running = 0
        best_suffix = float("-inf")
        for index in range(middle - 1, low - 1, -1):
            running += values[index]
            best_suffix = max(best_suffix, running)

        # Best prefix starting exactly at the midpoint.
        running = 0
        best_prefix = float("-inf")
        for index in range(middle, high):
            running += values[index]
            best_prefix = max(best_prefix, running)

        best_crossing = int(best_suffix + best_prefix)
        return max(best_left, best_right, best_crossing)

    return solve(0, len(values))


assert maximum_subarray_divide_conquer(
    [-2, 1, -3, 4, -1, 2, 1, -5, 4]
) == 6  # [4, -1, 2, 1]
assert maximum_subarray_divide_conquer([-4, -2, -7]) == -2
~~~

</details>

The recurrence is $T(n)=2T(n/2)+\Theta(n)=\Theta(n\log n)$. Kadane's algorithm solves the same problem in $\Theta(n)$ time, so this example illustrates the design pattern rather than the best specialized solution.

**Practice.** [LeetCode 53 - Maximum Subarray](https://leetcode.com/problems/maximum-subarray/) allows direct comparison between the divide-and-conquer formulation and the linear dynamic-programming invariant.


### **Recurrence Analysis** {#recurrence-analysis}

A **recurrence** defines an algorithm's cost on size <code>n</code> in terms of costs on smaller inputs. It is a model of the recursive code, not a formula chosen after seeing the desired answer. Every recurrence should identify:

- the base-case cost;
- the number and sizes of recursive calls;
- the nonrecursive division, partition, or combination work;
- whether floors, ceilings, or uneven sizes matter asymptotically.

Examples from this chapter are:

$$
\begin{aligned}
\text{binary search: } &T(n)=T(n/2)+\Theta(1),\\
\text{merge sort: } &T(n)=2T(n/2)+\Theta(n),\\
\text{balanced quicksort: } &T(n)=2T(n/2)+\Theta(n),\\
\text{extreme quicksort: } &T(n)=T(n-1)+\Theta(n).
\end{aligned}
$$

The first coefficient counts recursive calls. The argument gives each subproblem size. The final term counts work performed by the current call outside recursion.

Repeated substitution, also called **unrolling**, expands smaller terms until the base case appears. A recursion tree presents the same expansion by level and is often better for seeing where work accumulates.

![Expanding T(n)=2T(n/2)+n shows n work at each of log2(n) levels.](assets/recurrence-expansion.svg){fig-align="center" width="96%"}

For merge sort,

$$
T(n)=2T(n/2)+n
=4T(n/4)+2n
=8T(n/8)+3n.
$$

After <code>j</code> expansions there are $2^j$ subproblems of size $n/2^j$, while accumulated merge work is $jn$. Expansion stops when $n/2^j=1$, so $j=\log_2 n$. The leaves contribute $\Theta(n)$ and internal levels contribute $\Theta(n\log n)$, giving $T(n)=\Theta(n\log n)$.

Substitution can also prove a guessed bound by induction. To prove $T(n)=O(n\log n)$, assume smaller instances satisfy the proposed inequality, insert that assumption into the recurrence, and choose constants that make the result hold. The recursion tree discovers a likely bound; induction verifies it rigorously.

<details>
<summary>Python implementation: measure the merge-sort recurrence directly</summary>

~~~python
import math


def merge_recurrence_work(size: int) -> int:
    """Count unit base work plus size units at every merge call."""
    if size <= 1:
        return 1

    left_size = size // 2
    right_size = size - left_size
    return (
        merge_recurrence_work(left_size)
        + merge_recurrence_work(right_size)
        + size
    )


# For powers of two with T(1)=1 and merge cost n:
# T(n) = n + n log2(n).
for size in (1, 2, 4, 8, 16, 32):
    expected = size * (1 + int(math.log2(size)))
    assert merge_recurrence_work(size) == expected
~~~

</details>

The recursive measurement is educational rather than an efficient analyzer; it mirrors the call tree so the counted work can be matched to the algebra.

**Practice.** [LeetCode 50 - Pow(x, n)](https://leetcode.com/problems/powx-n/) uses exponent halving and leads to $T(n)=T(n/2)+\Theta(1)=\Theta(\log n)$ when the half-power is computed once and reused.


### **Recursion Trees and the Master Theorem** {#recursion-trees-and-the-master-theorem}

For recurrences of the form

$$
T(n)=aT(n/b)+f(n),
$$

the **Master Theorem** compares the current call's nonrecursive work $f(n)$ with the recursive threshold

$$
n^{\log_b a}.
$$

The parameter <code>a</code> is the number of subproblems, <code>b</code> is the factor by which each subproblem shrinks, and $p=\log_b a$ describes how the number of recursion-tree leaves grows. The theorem asks whether work is concentrated near the leaves, balanced across levels, or concentrated near the root.

![The three Master Theorem cases compare f(n) with the recursive threshold n raised to log base b of a.](assets/master-theorem-cases.svg){fig-align="center" width="96%"}

Using $p=\log_b a$ and a positive constant $\varepsilon$:

1. If $f(n)=O(n^{p-\varepsilon})$, recursive leaves dominate and $T(n)=\Theta(n^p)$.
2. If $f(n)=\Theta(n^p\log^k n)$ for $k\ge 0$, levels balance and $T(n)=\Theta(n^p\log^{k+1}n)$.
3. If $f(n)=\Omega(n^{p+\varepsilon})$ and the regularity condition $af(n/b)\le c f(n)$ holds for some $c<1$, root-side work dominates and $T(n)=\Theta(f(n))$.

For example:

| Recurrence | Comparison | Case | Bound |
|---|---|---:|---:|
| $T(n)=8T(n/2)+n^2$ | $p=3$ and $n^2$ is polynomially smaller | 1 | $\Theta(n^3)$ |
| $T(n)=2T(n/2)+n$ | $p=1$ and $f(n)=\Theta(n^p)$ | 2 | $\Theta(n\log n)$ |
| $T(n)=2T(n/2)+n^2$ | $p=1$ and $n^2$ is polynomially larger | 3 | $\Theta(n^2)$ |

The theorem is not universal. It does not directly handle $T(n)=T(n/3)+T(2n/3)+n$, subproblem counts that vary with <code>n</code>, or extreme quicksort's $T(n)=T(n-1)+n$. A recursion tree, substitution, Akra-Bazzi, or another technique is needed when the required form or side conditions fail.

<details>
<summary>Python implementation: classify the polynomial Master cases</summary>

~~~python
import math


def polynomial_master_case(a: float, b: float, degree: float) -> tuple[int, str]:
    """Classify T(n)=aT(n/b)+Theta(n^degree), with no log factor."""
    if a < 1 or b <= 1:
        raise ValueError("require a >= 1 and b > 1")

    threshold = math.log(a, b)          # p = log_b(a)
    tolerance = 1e-12

    if degree < threshold - tolerance:
        return 1, f"Theta(n^{threshold:g})"
    if degree > threshold + tolerance:
        # Polynomial f(n)=n^degree satisfies the regularity condition here.
        return 3, f"Theta(n^{degree:g})"
    return 2, f"Theta(n^{threshold:g} log n)"


assert polynomial_master_case(8, 2, 2) == (1, "Theta(n^3)")
assert polynomial_master_case(2, 2, 1) == (2, "Theta(n^1 log n)")
assert polynomial_master_case(2, 2, 2) == (3, "Theta(n^2)")
~~~

</details>

The helper deliberately handles only $f(n)=\Theta(n^d)$. Treating it as a general recurrence solver would hide logarithmic factors and required regularity conditions.

**Practice.** [LeetCode 169 - Majority Element](https://leetcode.com/problems/majority-element/) admits a divide-and-conquer solution whose two half-results are combined with linear counting work, providing a concrete $2T(n/2)+\Theta(n)$ recurrence.


### **Quickselect and Randomized Pivots** {#quickselect-and-randomized-pivots}

An **order statistic** is the item of a requested rank, such as the minimum, median, or <code>k</code>-th smallest value. Sorting the entire array gives the answer in $O(n\log n)$ time, but it computes more order information than one rank requires.

**Quickselect** uses the same partition step as quicksort. After partitioning, the pivot or equal-pivot region is in its final rank range. Comparing <code>k</code> with that range determines whether the answer is already found or lies entirely on one side. The other side can be discarded without sorting.

~~~text
QUICKSELECT(A, k)
    low <- 0; high <- length(A) - 1
    while low <= high
        choose a random pivot from A[low:high+1]
        lt, gt <- three-way partition around pivot
        if k < lt
            high <- lt - 1
        else if k > gt
            low <- gt + 1
        else
            return A[k]
~~~

![Quickselect compares the requested rank with the pivot rank and discards the side that cannot contain the answer.](assets/quickselect-shrink.svg){fig-align="center" width="96%"}

One partition scans the active interval in $\Theta(n)$ time. If random pivots produce a geometrically shrinking sequence of expected interval sizes, total expected work resembles

$$
n+\frac{n}{2}+\frac{n}{4}+\cdots=O(n).
$$

The exact expected analysis accounts for all pivot ranks, but the geometric sum gives the central intuition. Repeatedly choosing an extreme pivot produces $n+(n-1)+\cdots+1=\Theta(n^2)$ worst-case time. Random pivots make this behavior unlikely against fixed input order but do not remove the theoretical worst case.

An iterative in-place implementation uses $O(1)$ auxiliary storage. Three-way partitioning is valuable for duplicate-heavy input because the entire equal region can terminate the search at once.

<details>
<summary>Python implementation: randomized in-place Quickselect</summary>

~~~python
from random import Random


def quickselect(values: list[int], k: int, seed: int = 0) -> int:
    """Return the zero-based k-th smallest value, rearranging values in place."""
    if k < 0 or k >= len(values):
        raise IndexError("rank k is outside the array")

    random = Random(seed)
    low = 0
    high = len(values) - 1

    while low <= high:
        pivot = values[random.randrange(low, high + 1)]
        less = low
        scan = low
        greater = high

        while scan <= greater:
            if values[scan] < pivot:
                values[less], values[scan] = values[scan], values[less]
                less += 1
                scan += 1
            elif values[scan] > pivot:
                values[scan], values[greater] = values[greater], values[scan]
                greater -= 1
            else:
                scan += 1

        if k < less:
            high = less - 1              # Discard pivot and larger region.
        elif k > greater:
            low = greater + 1            # Discard pivot and smaller region.
        else:
            return values[k]

    raise RuntimeError("unreachable for a valid rank")


sample = [9, 1, 8, 2, 7, 3, 6, 4, 5, 5]
assert quickselect(sample.copy(), 0, seed=7) == 1
assert quickselect(sample.copy(), 4, seed=7) == 5
assert quickselect(sample.copy(), 9, seed=7) == 9
~~~

</details>

**Practice.** [LeetCode 215 - Kth Largest Element in an Array](https://leetcode.com/problems/kth-largest-element-in-an-array/) is the direct Quickselect exercise after converting a largest-rank request into the corresponding zero-based smallest rank.


### **Comparison and Selection** {#comparison-and-selection}

The right technique depends on how much order information is needed and which guarantees the application values.

| Goal or constraint | Recommended starting point | Time | Auxiliary space | Important property |
|---|---|---:|---:|---|
| exact lookup in sorted random-access data | binary search | $O(\log n)$ | $O(1)$ | returns any match unless boundary logic is added |
| first feasible value or duplicate boundary | first-true / lower-bound template | $O(P\log R)$ | $O(1)$ | predicate must be monotone |
| tiny or nearly sorted input | insertion sort | best $O(n)$, worst $O(n^2)$ | $O(1)$ | stable and adaptive |
| stable predictable array sorting | merge sort | $\Theta(n\log n)$ | $\Theta(n)$ | stable, sequential merge access |
| fast typical in-place array sorting | randomized three-way quicksort | expected $\Theta(n\log n)$ | expected $O(\log n)$ stack | strong locality, unstable |
| in-place worst-case sorting guarantee | heapsort | $\Theta(n\log n)$ | $O(1)$ | unstable, weaker locality |
| one rank rather than total order | randomized Quickselect | expected $\Theta(n)$ | $O(1)$ iterative | worst case $\Theta(n^2)$ |
| non-comparison integer keys with small range | counting / radix family | model-dependent, often $O(n+k)$ | model-dependent | escapes comparison lower bound |

A disciplined workflow is:

1. State whether the input is already sorted or whether a monotone predicate exists.
2. Decide whether the output requires one item, a boundary, partial order, or complete order.
3. Specify stability, mutation, memory, and worst-case requirements before selecting a sort.
4. For recursive algorithms, write the base case and map every recurrence term to code.
5. Check theorem assumptions before quoting a bound; unequal quicksort subproblems do not fit the basic Master Theorem.
6. Test empty input, one item, duplicates, sorted and reverse-sorted input, and highly unbalanced partitions.

The unifying idea is **safe elimination**. Binary search removes impossible indices, sorting fixes progressively more order, divide-and-conquer summarizes completed subproblems, and Quickselect discards ranks that cannot contain the requested answer.

**Practice.** [LeetCode 4 - Median of Two Sorted Arrays](https://leetcode.com/problems/median-of-two-sorted-arrays/) is a demanding synthesis problem: it searches a partition boundary rather than merging or sorting all values.
